# Tutorial 11 — Interpretability and cross-lingual steering

[Lecture notes](../lectures/11_steering.md) · [Slides](../slides/11_steering.pptx) · [Exercise sheet](../exercises/11_steering.md)

**40 minutes, CPU, NumPy only.** You will find a language direction in a toy residual stream, steer English activations into Swahili, and measure what steering costs. The activations are synthetic: 32 numbers per example, built so that you know the right answer. Nothing here measures a real model.

| Minutes | Step |
|---|---|
| 0–5 | Build the toy residual stream |
| 5–15 | Read: difference of means and a control direction |
| 15–25 | Write: steer and sweep the strength α |
| 25–30 | Remove: directional ablation |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. A toy residual stream

Each activation is a sum of three parts: a topic direction (4 topics), a language direction (English −1.5, Swahili +1.5) and noise. The language direction `u` leans 0.6 towards topic 0. Real features overlap like this, and the overlap is what makes steering costly.

The "rest of the model" is two fixed readouts. `lang_of` reads the output language. `topic_of` reads what the answer is about. A steered answer is useful only if both are right.

In [2]:
d, K = 32, 4
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))      # orthonormal rows
topic_dirs, read = Q[:K], Q[K]
u = Q[K] + 0.6 * Q[0]; u /= np.linalg.norm(u)     # language direction, entangled with topic 0

def activations(lang, n_per_topic):
    topic = np.repeat(np.arange(K), n_per_topic)
    h = 3 * topic_dirs[topic] + (1.5 if lang else -1.5) * u + rng.normal(0, 0.6, (len(topic), d))
    return h, topic

def lang_of(h):  return (h @ read > 0).astype(int)          # 0 = English, 1 = Swahili
def topic_of(h): return (h @ topic_dirs.T).argmax(1)

H_en, _ = activations(0, 200); H_sw, _ = activations(1, 200)      # used to find the direction
D_en, t_en = activations(0, 100); D_sw, t_sw = activations(1, 100)  # held out, used to score
print("held-out English read as Swahili:", lang_of(D_en).mean())
print("held-out topic accuracy:", (topic_of(D_en) == t_en).mean())

held-out English read as Swahili: 0.0225
held-out topic accuracy: 1.0


## 2. Read: difference of means

The steering vector is the mean Swahili activation minus the mean English activation. Before you run the cell, predict the cosine between `v` and the true direction `u`, and the accuracy of a probe along a random direction.

In [3]:
v = H_sw.mean(0) - H_en.mean(0)
v_hat = v / np.linalg.norm(v)
both = np.r_[D_en, D_sw]; is_sw = np.r_[np.zeros(len(D_en)), np.ones(len(D_sw))]

def probe_accuracy(direction):
    score = both @ direction
    return ((score > score.mean()) == is_sw).mean()

random_dir = rng.normal(size=d); random_dir /= np.linalg.norm(random_dir)
print("cosine(v, u) =", round(float(v_hat @ u), 3), " |v| =", round(float(np.linalg.norm(v)), 2))
print("probe along v:", probe_accuracy(v_hat), " probe along a random direction:", probe_accuracy(random_dir))
assert v_hat @ u > 0.95 and probe_accuracy(v_hat) > 0.95

cosine(v, u) = 0.997  |v| = 3.04
probe along v: 0.9525  probe along a random direction: 0.60125


The probe along `v` separates the languages. That shows the information is there. It does not show that the model uses it. Section 3 tests use by writing to the direction.

## 3. Write: steer and sweep α

Add `α·v` to every held-out English activation. **Compliance** is the share read as Swahili. **Relevance** is the share whose topic is still right. The course combines them with a harmonic mean so that neither can be traded away; the Task 7 page names both criteria and does not publish a formula.

Predict the shape of each column before running.

In [4]:
def score(h, topics):
    compliance = lang_of(h).mean()
    relevance = (topic_of(h) == topics).mean()
    return compliance, relevance, 2 * compliance * relevance / (compliance + relevance + 1e-12)

alphas = [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4]
print(" alpha  compliance  relevance  harmonic  mean |h|")
sweep = []
for a in alphas:
    h = D_en + a * v
    c, r, f = score(h, t_en)
    sweep.append((a, c, r, f))
    print(f"{a:6.2f}  {c:10.3f}  {r:9.3f}  {f:8.3f}  {np.linalg.norm(h, axis=1).mean():8.2f}")
best = max(sweep, key=lambda row: row[3])
print("best alpha on this set:", best[0])
assert sweep[0][1] < 0.1 and sweep[-1][1] == 1.0     # compliance rises with alpha
assert sweep[-1][2] < 0.5                            # and relevance collapses when alpha is too large

 alpha  compliance  relevance  harmonic  mean |h|
  0.00       0.022      1.000     0.044      4.62
  0.25       0.160      1.000     0.276      4.51
  0.50       0.527      1.000     0.691      4.52
  0.75       0.885      1.000     0.939      4.65
  1.00       0.993      0.998     0.995      4.89
  1.50       1.000      0.970     0.985      5.66
  2.00       1.000      0.812     0.897      6.69
  3.00       1.000      0.352     0.521      9.17
  4.00       1.000      0.253     0.403     11.92
best alpha on this set: 1


Three things to notice.

1. Below α = 1 the answer stays in English. At α = 1 the activation has moved one full mean difference and compliance is near 1.
2. Past α = 1.5 relevance falls. `v` carries a piece of topic 0, so a large push drags every answer towards topic 0. The last column shows the activation norm growing away from anything the readouts saw in training.
3. `best alpha` was chosen on the same examples it is scored on. In your Task 7 system, choose the layer and α on development questions and report the test score once.

## 4. Remove: directional ablation

Project the direction out: `h − (h·v̂)v̂`. If the language readout depends on this direction, language accuracy should fall to chance while topic accuracy stays.

In [5]:
ablated = both - np.outer(both @ v_hat, v_hat)
lang_acc = (lang_of(ablated) == is_sw).mean()
topic_acc = (topic_of(ablated) == np.r_[t_en, t_sw]).mean()
print("after ablation: language accuracy", lang_acc, " topic accuracy", topic_acc)
assert abs(lang_acc - 0.5) < 0.1 and topic_acc > 0.95

after ablation: language accuracy 0.5225  topic accuracy 0.9975


Language accuracy is at chance and topic accuracy is untouched. Reading (section 2), writing (section 3) and removing (section 4) agree, which is the evidence pattern to aim for in a paper: a probe alone is the weakest of the three.

## 5. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Difference of means for any pair of labels.** Task 7 has 20 languages, so you need the vector for any (source, target) pair from one labelled activation matrix.

In [ ]:
# YOUR TURN
def steering_vector(H, labels, source, target):
    """Mean activation of rows labelled `target` minus mean of rows labelled `source`."""
    raise NotImplementedError("your turn")

In [7]:
H_all = np.r_[H_en, H_sw]; labels = np.array(["en"] * len(H_en) + ["sw"] * len(H_sw))
assert np.allclose(steering_vector(H_all, labels, "en", "sw"), v)
assert np.allclose(steering_vector(H_all, labels, "sw", "en"), -v)
print("A passed")

A passed


**B. Clamp, do not add.** Additive steering moves every example by the same amount, whatever its starting point. Clamping sets each example's projection on `v̂` to a target value and leaves the rest of the vector alone. Use the mean projection of the Swahili training activations as the target.

In [ ]:
# YOUR TURN
def clamp(h, direction, target):
    """Return h with its projection on the unit vector `direction` set to `target` for every row."""
    raise NotImplementedError("your turn")

In [9]:
target = float((H_sw @ v_hat).mean())
clamped = clamp(D_en, v_hat, target)
assert np.allclose(clamped @ v_hat, target)                                   # projection is exactly the target
assert np.allclose(clamped - np.outer(clamped @ v_hat, v_hat), D_en - np.outer(D_en @ v_hat, v_hat))  # the rest is untouched
c, r, f = score(clamped, t_en)
print(f"clamp: compliance {c:.3f}  relevance {r:.3f}  harmonic {f:.3f}")
print("B passed")

clamp: compliance 0.975  relevance 0.995  harmonic 0.985
B passed


## Discussion

1. A probe finds language identity at 99% accuracy in layer 2, but steering at layer 2 changes nothing. Give two explanations and one experiment that separates them.
2. Clamping needs no α. What did you have to choose instead, and on which data?
3. The toy readout is a fixed vector. Name two ways a real model's later layers could undo your steering.

## From the toy to Task 7

The organisers' repository holds FLORES-200 development sentences, the CLaS-Bench validation set, a baseline and the scorer: <https://github.com/d-gurgurov/CLaS-SemEval-2027>. Start from their code. The steps are the ones above: collect activations at one layer for parallel sentences, take the difference of means, add it during generation with a forward hook, and sweep the layer and α on development questions. Prompt-based steering is not ranked.

Submit this notebook with both tests passing and a 150–250-word note: the α you would choose, the evidence for it, and one way the toy flatters the method.